## Setup

---

In [ ]:
! pip install transformers
! pip install datasets
! pip install sentencepiece
! pip install rouge_score
! pip install wandb

In [ ]:
import torch
import numpy as np
import datasets

from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)

from tabulate import tabulate
import nltk
from datetime import datetime

In [ ]:
WANDB_INTEGRATION = True
if WANDB_INTEGRATION:
    import wandb

    wandb.login()

## Model and tokenizer

---

Download model and tokenizer. Use default parameters or try custom values (see [HF Bart configuration](https://huggingface.co/transformers/_modules/transformers/configuration_bart.html) and [Fairseq Bart](https://github.com/pytorch/fairseq/tree/master/examples/bart)).

In [ ]:
model_name = "facebook/mbart-large-50-many-to-many-mmt"


tokenizer = AutoTokenizer.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")

# Set model parameters or use the default
# print(model.config)

# tokenization
encoder_max_length = 512  # demo
decoder_max_length = 150

## Data

---

### Download

For demonstration, we are only using a small portion of the data.

In [ ]:
# prompt: hugging face login with tokens given

!huggingface-cli login --token REDACTED_CONFIGURE_TOKEN_OUTSIDE_SOURCE

In [ ]:
from datasets import load_dataset

# Load the Tamil summarization dataset
language = "tamil"
data = load_dataset("HariprasathSB/tamil_summarization", split="train[:3000]")



### Prepare

**Format and split into train and validation sets**

In [ ]:
dataset = data.map(batched=True)

train_data_txt, validation_data_txt = dataset.train_test_split(test_size=0.1).values()

**Preprocess and tokenize**

In [ ]:
def batch_tokenize_preprocess(batch, tokenizer, max_source_length, max_target_length):
    source, target = batch["Text"], batch["Summary"]
    source_tokenized = tokenizer(
        source, padding="max_length", truncation=True, max_length=max_source_length
    )
    target_tokenized = tokenizer(
        target, padding="max_length", truncation=True, max_length=max_target_length
    )

    batch = {k: v for k, v in source_tokenized.items()}
    # Ignore padding in the loss
    batch["labels"] = [
        [-100 if token == tokenizer.pad_token_id else token for token in l]
        for l in target_tokenized["input_ids"]
    ]
    return batch

# Modify the data mapping and preprocessing functions accordingly
dataset = dataset.map()

train_data_txt, validation_data_txt = dataset.train_test_split(test_size=0.1).values()

# Assuming you have a tokenizer and encoder_max_length, decoder_max_length defined
train_data = train_data_txt.map(
    lambda batch: batch_tokenize_preprocess(
        batch, tokenizer, encoder_max_length, decoder_max_length
    ),
    batched=True,
)

validation_data = validation_data_txt.map(
    lambda batch: batch_tokenize_preprocess(
        batch, tokenizer, encoder_max_length, decoder_max_length
    ),
    batched=True,
)


## Training

---

### Metrics

In [ ]:
# Borrowed from https://github.com/huggingface/transformers/blob/master/examples/seq2seq/run_summarization.py

nltk.download("punkt", quiet=True)

metric = datasets.load_metric("rouge")


def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [label.strip() for label in labels]

    # rougeLSum expects newline after each sentence
    preds = ["\n".join(nltk.sent_tokenize(pred)) for pred in preds]
    labels = ["\n".join(nltk.sent_tokenize(label)) for label in labels]

    return preds, labels


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Some simple post-processing
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(
        predictions=decoded_preds, references=decoded_labels, use_stemmer=True
    )
    # Extract a few results from ROUGE
    result = {key: value.mid.fmeasure * 100 for key, value in result.items()}

    prediction_lens = [
        np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds
    ]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result

### Training arguments

In [ ]:
!pip install accelerate -U

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="results",
    num_train_epochs=5,  # Adjusted
    do_train=True,
    do_eval=True,
    per_device_train_batch_size=16,  # Adjusted
    per_device_eval_batch_size=16,  # Adjusted
    learning_rate=5e-05,  # Adjusted
    warmup_steps=1000,  # Adjusted
    weight_decay=0.01,  # Adjusted
    label_smoothing_factor=0.1,
    predict_with_generate=True,
    logging_dir="logs",
    logging_steps=200,  # Adjusted
    save_total_limit=2,  # Adjusted
)

# Data collator for seq2seq tasks
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Define the trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_data,
    eval_dataset=validation_data,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

### Train

Wandb integration

In [ ]:
if WANDB_INTEGRATION:
    wandb_run = wandb.init(
        project="bart_tam_sum",
        config={
            "per_device_train_batch_size": training_args.per_device_train_batch_size,
            "learning_rate": training_args.learning_rate,
            "dataset": "wiki_lingua " + language,
        },
    )

    now = datetime.now()
    current_time = now.strftime("%H%M%S")
    wandb_run.name = "run_" + language + "_" + current_time

Evaluate before fine-tuning

In [ ]:
trainer.evaluate()

Train the model

In [ ]:
#%%wandb
# uncomment to display Wandb charts

trainer.train()

Evaluate after fine-tuning

In [ ]:
trainer.evaluate()

In [ ]:
%%wandb

if WANDB_INTEGRATION:
    wandb_run.finish()

## Evaluation

---

**Generate summaries from the fine-tuned model and compare them with those generated from the original, pre-trained one.**

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
model_name = "facebook/mbart-large-50-many-to-many-mmt"

def generate_summary(test_samples, model, tokenizer, encoder_max_length):
    inputs = tokenizer(
        test_samples["Text"],
        padding="max_length",
        truncation=True,
        max_length=encoder_max_length,
        return_tensors="pt",
    )
    input_ids = inputs.input_ids.to(model.device)
    attention_mask = inputs.attention_mask.to(model.device)
    outputs = model.generate(input_ids, attention_mask=attention_mask)
    output_str = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return outputs, output_str

#Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_before_tuning = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Select test samples from your dataset
test_samples = validation_data_txt.select(range(16))

# Generate summaries
encoder_max_length = 512  # Set your desired max length
summaries_before_tuning = generate_summary(test_samples, model_before_tuning, tokenizer, encoder_max_length)[1]
summaries_after_tuning = generate_summary(test_samples, model, tokenizer, encoder_max_length)[1]

In [ ]:
print(
    tabulate(
        zip(
            range(len(summaries_after_tuning)),
            summaries_after_tuning,
            summaries_before_tuning,
        ),
        headers=["Id", "Summary after", "Summary before"],
    )
)
print("\nTarget summaries:\n")
print(
    tabulate(list(enumerate(test_samples["summary"])), headers=["Id", "Target summary"])
)
print("\nSource documents:\n")
print(tabulate(list(enumerate(test_samples["document"])), headers=["Id", "Document"]))

## Examples

It appears that for both languages, after a short fine-tuning, our model has adopted the imperative mode of target summaries.